---------------------------------------------------------

In [ ]:
import custom_variables as params
import os
import json
import pandas as pd
from padwh.connectors.sharepoint import OnlineSharepointConnector
from global_variables import process_dir, env
from logger_singleton import logger, LogLevel, LogStatus, collect_db_logs

In [ ]:
# try:
step_name="Main"
logger.log(log_level=LogLevel.INFO, status=LogStatus.START, step_name=step_name, message=f"Process started.")

cert_path = os.path.join(process_dir, 'selfsigncert.pem')

with open(cert_path, "w") as f:
    f.write(params.cert)

cert_credentials = {
    "tenant": params.tenant,
    "client_id": params.appId,
    "thumbprint": params.thumbprint,
    "cert_path": cert_path
}

    
sp = OnlineSharepointConnector(site_url=params.siteUrl, rel_url=params.relUrl)
logger.log(log_level=LogLevel.INFO, status=LogStatus.START, step_name=step_name, message=f"Connecting to Sharepoint")

sp.connect_via_cert(**cert_credentials)
logger.log(log_level=LogLevel.INFO, status=LogStatus.START, step_name=step_name, message=f"Connection to Sharepoint - OK")

Time: 2026-05-11 12:30:15 | MasterPipelineName: my_GRANDSLAM_SHAREPOINT | LogLevel: info| Status: Start | StepName: Main | Message: Process started. | Metrics: {} | CustomDimensions: {}
Time: 2026-05-11 12:30:24 | MasterPipelineName: my_GRANDSLAM_SHAREPOINT | LogLevel: info| Status: Start | StepName: Main | Message: Connecting to Sharepoint | Metrics: {} | CustomDimensions: {}
Time: 2026-05-11 12:30:26 | MasterPipelineName: my_GRANDSLAM_SHAREPOINT | LogLevel: info| Status: Start | StepName: Main | Message: Connection to Sharepoint - OK | Metrics: {} | CustomDimensions: {}


In [ ]:
import io
from concurrent.futures import ThreadPoolExecutor, as_completed
from tenacity import retry, retry_if_not_exception_type, stop_after_attempt, wait_exponential
from typing import Any, Dict
from office365.sharepoint.listitems.listitem import ListItem

import hashlib
import re
from datetime import datetime, timezone
from global_variables import data_container, metadata_table

MAX_WORKERS = 10
PROGRESS_LOG_INTERVAL = 500
_INVALID_KEY_CHARS = re.compile(r"[\\/#?\x00-\x1f\x7f-\x9f]")

class NonRetryableError(Exception):
    """Marks validation errors that should not be retried."""

def _build_blob_name(server_relative_url: str) -> str:
    blob_name = server_relative_url.strip().replace("\\", "/").replace("/sites/OnePCP-DMS/", "") 
    if not blob_name:
        raise NonRetryableError("SharePoint item has an empty FileRef value.")
    return blob_name



@retry(
    wait=wait_exponential(multiplier=2, min=2, max=60),
    stop=stop_after_attempt(5),
    retry=retry_if_not_exception_type(NonRetryableError),
    reraise=True,
)
def download_and_upload(item: Dict[str, Any], cert_credentials: Dict[str, str]) -> str:
    """Downloads one SharePoint file into memory and uploads it to Blob storage."""
    server_relative_url = item.get("FileRef")
    if not server_relative_url:
        raise NonRetryableError("SharePoint item is missing FileRef.")

    blob_name = _build_blob_name(server_relative_url)
    step_name = f"Processing {blob_name}"

    # TODO: Address the .url files instead of skipping them
    # if file ends with .url, skip it as it's a shortcut file 
    if server_relative_url.lower().endswith(".url"):
        logger.log(
            log_level=LogLevel.INFO,
            status=LogStatus.SUCCESS,
            step_name=step_name,
            message=f"Skipped .url file: {server_relative_url}",
        )
        return blob_name

    sp = OnlineSharepointConnector(site_url=params.siteUrl, rel_url=params.relUrl)
    sp.connect_via_cert(**cert_credentials)

    try:
        with io.BytesIO() as file_buffer:
            file_obj = sp.ctx.web.get_file_by_server_relative_url(server_relative_url)
            file_obj.download(file_buffer).execute_query()
            file_buffer.seek(0)

            data_container.upload_blob(
                name=blob_name,
                data=file_buffer,
                # metadata=_build_blob_metadata(item),
                overwrite=True,
            )

            # TODO: upload metadata to table storage
            # metadata_table.upsert_entity(_build_table_entity(item, blob_name))
    except Exception as exc:
        logger.log(
            log_level=LogLevel.ERROR,
            status=LogStatus.ERROR,
            step_name=step_name,
            message=f"Failed to transfer file {server_relative_url}. Error: {exc}",
        )
        raise

    logger.log(
        log_level=LogLevel.INFO,
        status=LogStatus.SUCCESS,
        step_name=step_name,
        message=f"Uploaded to blob: {blob_name}",
    )
    return blob_name

def print_progress(items):    
    print("Items read: {0}".format(len(items)))

large_list = sp.ctx.web.lists.get_by_title(params.listName)
select_fields = ["FileRef", "FileDirRef", "FileLeafRef", "FileSystemObjectType"] 
select_fields.extend(["ABB_Coll_LifecycleStatus", "NextECM_Mig_File_Ext", "ABB_Coll_ApprovalDate", "ABB_Coll_ApprovedByPerson", "ABB_Coll_DocumentId", 
                      "ABB_Coll_DocumentKind", "ABB_Coll_DocumentPartID", "ABB_Coll_DocumentRevisionId", "ABB_Coll_LanguageCode", "DMS_LanguageCode", 
                      "ABB_Coll_Lifecycle", "ABB_Coll_OwningOrganization", "ABB_Coll_PreparedByPerson", "ABB_Coll_PreparedDate", "ABB_Coll_ResponsibleDepartment", 
                      "ABB_Coll_SecurityLevel", "SupplementaryTitle", "DMS_TAG", "Title", "ABB_Coll_TitleEnglish"])

all_items = (
    large_list.items
    .select(select_fields)    
    .get_all(5000, print_progress)
    .execute_query()
)
print("Total items count: {0}".format(len(all_items)))

# 2. Filter the items in Python memory
ALLOWED_EXTENSIONS = {"pdf"}#, "doc", "docx"}
filtered_ListItems = [item for item in all_items if item.properties["FileSystemObjectType"] == 0 and item.properties["NextECM_Mig_File_Ext"] in ALLOWED_EXTENSIONS and item.properties["ABB_Coll_LifecycleStatus"] == "Approved"]
# filtered_items = [item.properties for item in filtered_ListItems]

success_count = 0
fail_count = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(download_and_upload, item.properties, cert_credentials): item.properties
        for item in filtered_ListItems
    }

    for future in as_completed(futures):
        item = futures[future]
        file_ref = str(item.get("FileRef", "<missing FileRef>"))
        try:
            future.result()
            success_count += 1
            if success_count % PROGRESS_LOG_INTERVAL == 0 or success_count == len(filtered_ListItems):
                logger.log(
                    log_level=LogLevel.INFO,
                    status=LogStatus.START,
                    step_name=step_name,
                    message=f"Progress: {success_count}/{len(filtered_ListItems)} files uploaded.",
                )
        except Exception as exc:
            fail_count += 1
            logger.log(
                log_level=LogLevel.ERROR,
                status=LogStatus.ERROR,
                step_name=step_name,
                message=f"Failed to process {file_ref}. Error: {exc}",
            )


--------------------------------------------------------------------------

In [15]:
print("Items in filtered list:", len(filtered_ListItems))
filtered_ListItems[0].properties

Items in filtered list: 3330


{'ParentList': SP.List,
 'FileSystemObjectType': 0,
 'FileLeafRef': '2PAA122125_en Review Protocol CIT 6.1.1.pdf',
 'Title': 'Review Protocol CIT 6.1.1',
 'ABB_Coll_DocumentId': '2PAA122125',
 'ABB_Coll_DocumentKind': {'Label': '72',
  'TermGuid': '65351595-3284-41ed-80f2-b49ede3c8e8e',
  'WssId': 72},
 'ABB_Coll_DocumentRevisionId': 'A',
 'ABB_Coll_LifecycleStatus': 'Approved',
 'ABB_Coll_LanguageCode': {'Label': '8',
  'TermGuid': '645eeacf-9e24-46f0-b6e5-4c5343c3dda6',
  'WssId': 8},
 'ABB_Coll_PreparedByPerson': 'Fredrik Rosdahl',
 'ABB_Coll_PreparedDate': '2021-05-24T22:00:00Z',
 'ABB_Coll_SecurityLevel': 'Internal',
 'ABB_Coll_ApprovalDate': '2021-05-24T22:00:00Z',
 'ABB_Coll_ApprovedByPerson': 'Fredrik  Rosdahl',
 'ABB_Coll_DocumentPartID': None,
 'ABB_Coll_OwningOrganization': 'IAPCP',
 'ABB_Coll_TitleEnglish': None,
 'DMS_LanguageCode': 'en',
 'SupplementaryTitle': 'Not Defined',
 'ABB_Coll_ResponsibleDepartment': None,
 'NextECM_Mig_File_Ext': 'pdf',
 'ABB_Coll_Lifecycle': 'I

In [24]:
with io.BytesIO() as file_buffer:
    file_obj = sp.ctx.web.get_file_by_server_relative_url("/sites/OnePCP-DMS/Mig_10_Operations/04 Product Development and Release/01 800xA/Control/01 Technical documents/Safety Certificate/Certificate/3BSE080872_C15310014_BST_Confirmation_of_BurnerLibrary.url")
    file_obj.download(file_buffer).execute_query()
    file_buffer.seek(0)

    # save to local file for testing
    with open("test_downloaded_file.url", "wb") as f:
        f.write(file_buffer.read())

In [34]:
# find item from the list with filename '3BNP100734.pdf' and print its properties
for item in filtered_ListItems:
    if item.properties.get("FileLeafRef", "") == "3BSE080872_C15310014_BST_Confirmation_of_BurnerLibrary (1).pdf":
        display(item.properties)
        print(item.properties.get("FileRef"))
        break

{'ParentList': SP.List,
 'FileSystemObjectType': 0,
 'FileLeafRef': '3BSE080872_C15310014_BST_Confirmation_of_BurnerLibrary (1).pdf',
 'Title': 'AC 800M Burner Library 1.0-3 Confirmation',
 'ABB_Coll_DocumentId': '3BSE080872',
 'ABB_Coll_DocumentKind': {'Label': '79',
  'TermGuid': '5a013460-12a4-4652-bdda-8d23c87baeb9',
  'WssId': 79},
 'ABB_Coll_DocumentRevisionId': '-',
 'ABB_Coll_LifecycleStatus': 'Approved',
 'ABB_Coll_LanguageCode': {'Label': '8',
  'TermGuid': '645eeacf-9e24-46f0-b6e5-4c5343c3dda6',
  'WssId': 8},
 'ABB_Coll_PreparedByPerson': 'Thilderqvist Annika',
 'ABB_Coll_PreparedDate': '2014-09-17T22:00:00Z',
 'ABB_Coll_SecurityLevel': 'Internal',
 'ABB_Coll_ApprovalDate': '2014-12-01T23:00:00Z',
 'ABB_Coll_ApprovedByPerson': 'Nordström Hans B',
 'ABB_Coll_DocumentPartID': None,
 'ABB_Coll_OwningOrganization': 'PACT ABB AB XAAC',
 'ABB_Coll_TitleEnglish': None,
 'DMS_LanguageCode': 'en',
 'SupplementaryTitle': 'Not Defined',
 'ABB_Coll_ResponsibleDepartment': 'XAAC',
 'Nex

/sites/OnePCP-DMS/Mig_10_Operations/04 Product Development and Release/01 800xA/Control/01 Technical documents/Reports/Certification/3BSE080872_C15310014_BST_Confirmation_of_BurnerLibrary (1).pdf


-----------------------------------------------------